In [1]:
import os
from helper.problem import examples
from chemistry_agent.env.env import ChemistryEnv
from chemistry_agent.agent.agent import GuidedAgent

os.environ["OPENAI_API_KEY"] = ""

In [2]:
agent = GuidedAgent()
MAX_CALLS = 5

for problem in examples:
    env = ChemistryEnv(problem)
    obs, tools = await env.reset()
    state = await agent.init_state(tools)

    print("\n--- Chain‐of‐Thought ---")
    for step in range(1, MAX_CALLS + 1):
        # 1) Ask the agent for its next move
        op_result, state, _ = await agent.get_asv(state, obs)
        action = op_result.value
        print(op_result)
        print(op_result.value)

        # 2) Execute in the env
        obs, _, _, _ = await env.step(action)
        tool_output = obs[0].content.strip()
        print("[Tool output]", tool_output)
        print(f"[Expected]: {problem.expected.strip()}")

        # 3) If the answer is here, stop
        if tool_output == problem.expected.strip():
            print("\n[Final answer]", tool_output)
            print("Expected:", problem.expected)
            print("Reward:", float(tool_output == problem.expected.strip()))
            break


[self.problem]: prompt=None task='molar_mass' params={'formula': 'H2O'} expected='18.02 g/mol'
[DEBUG build_prompt] task='molar_mass', params={'formula': 'H2O'}
[RESET] prompt: What is the molar mass of H2O?

--- Chain‐of‐Thought ---
[DEBUG]: [Message(role='system', content='You are a scientific assistant.  You **must** use the provided tools for *every* calculation.\nStep by step:\n  1) To compute a mixture’s concentration, call **mix_concentration** with JSON args.\n  2) Observe the tool’s output (e.g. “0.175 M”).\n  3) To compute dilution volume, call **dilution** with JSON args using that exact output.\n  4) Observe and return that final tool output as your answer.'), Message(role='user', content='What is the molar mass of H2O?')]


BadRequestError: litellm.BadRequestError: OpenAIException - You didn't provide an API key. You need to provide your API key in an Authorization header using Bearer auth (i.e. Authorization: Bearer YOUR_KEY), or as the password field (with blank username) if you're accessing the API from your browser and are prompted for a username and password. You can obtain an API key from https://platform.openai.com/account/api-keys.. Received Model Group=gpt-4o-2024-08-06
Available Model Group Fallbacks=None LiteLLM Retried: 2 times, LiteLLM Max Retries: 3